[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [SQLModel, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlmodel-deep-dive.html)

# Migrations &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

The first cell is the notebook's Setup and the project its worked examples built: `models.py`,
Alembic pointed at `SQLModel.metadata`, `import sqlmodel` in the template, and revision `0001`
applied to `scratch/heroes.db`. Run it first. The tasks change the same project in order, and the
last cell removes the scratch folder.


In [1]:
import os
import re
import shlex
import shutil
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

for package, pin in (("sqlmodel", "sqlmodel==0.0.42"), ("alembic", "alembic==1.20.0")):
    try:
        version(package)
    except PackageNotFoundError:                                    # Colab has neither: install the pinned versions
        subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--root-user-action=ignore", pin],
                       check=True)

import sqlmodel
from sqlalchemy import create_engine, inspect, text

SCRATCH = Path("scratch")


def alembic(*arguments):
    """Run one alembic command in the scratch folder and print what it said."""
    done = subprocess.run([sys.executable, "-m", "alembic", *arguments], cwd=SCRATCH,
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                          env={**os.environ, "NO_COLOR": "1", "PYTHONUNBUFFERED": "1",
                               "PYTHONDONTWRITEBYTECODE": "1"})
    lines = done.stdout.replace(f"{SCRATCH.resolve()}{os.sep}", "").splitlines()
    if "Traceback (most recent call last):" in lines:               # the error's own line, not the whole traceback
        named = [line for line in lines if re.match(r"[\w.]+(Error|Exception): ", line)]
        lines = lines[:lines.index("Traceback (most recent call last):")] + ["Traceback ...", named[-1]]
    print("$", shlex.join(["alembic", *arguments]))
    for line in lines:
        if not any(noise in line for noise in ("Context impl", "Will assume", "Please edit",
                                               "setting up autogenerate plugin")):
            print("   ", line)


def run_python(source, name="a_script.py"):
    """Write a small program into the scratch folder, run it in a Python of its own, and print what it said."""
    (SCRATCH / name).write_text(source)
    done = subprocess.run([sys.executable, name], cwd=SCRATCH, capture_output=True, text=True)
    printed = done.stdout.strip() or done.stderr.strip().splitlines()[-1]
    print(printed.replace(f"{SCRATCH.resolve()}{os.sep}", ""))


def edit(path, old, new):
    """Change one piece of a file, and fail rather than silently do nothing."""
    text = path.read_text()
    assert text.count(old) == 1, f"{path.name}: found {text.count(old)} of {old!r}"
    path.write_text(text.replace(old, new))


def revision(name):
    """A revision script from its def upgrade on: the header holds the time it was written."""
    body = (SCRATCH / "migrations" / "versions" / name).read_text()
    print(body[body.index("def upgrade"):body.index("def downgrade")].rstrip())


def columns_of(table, database="heroes.db"):
    """The column names a database has for a table, read from the database itself."""
    engine = create_engine(f"sqlite:///scratch/{database}")
    try:
        return [column["name"] for column in inspect(engine).get_columns(table)]
    finally:
        engine.dispose()

MODELS = """from sqlmodel import Field, Relationship, SQLModel


class Team(SQLModel, table=True):
    id: int | None = Field(default=None, primary_key=True)
    name: str = Field(index=True, max_length=50)
    headquarters: str = Field(max_length=60)

    heroes: list["Hero"] = Relationship(back_populates="team")


class Hero(SQLModel, table=True):
    id: int | None = Field(default=None, primary_key=True)
    name: str = Field(index=True, max_length=50)
    secret_name: str = Field(max_length=60)
    age: int | None = Field(default=None, index=True)
    team_id: int | None = Field(default=None, foreign_key="team.id")

    team: Team | None = Relationship(back_populates="heroes")
"""

shutil.rmtree(SCRATCH, ignore_errors=True)                          # a rerun starts from no project at all
SCRATCH.mkdir()
(SCRATCH / "models.py").write_text(MODELS)

print("sqlmodel", sqlmodel.__version__, "| alembic", version("alembic"),
      "| the project:", sorted(path.name for path in SCRATCH.iterdir()))


alembic("init", "migrations")
ini = SCRATCH / "alembic.ini"
ini.write_text(re.sub(r"^sqlalchemy\.url = .*$", "sqlalchemy.url = sqlite:///heroes.db",
                      ini.read_text(), flags=re.M))
edit(SCRATCH / "migrations" / "env.py", "target_metadata = None",
     'import sys\n\nsys.path.insert(0, ".")\nfrom models import SQLModel\n\ntarget_metadata = SQLModel.metadata')
edit(SCRATCH / "migrations" / "script.py.mako", "import sqlalchemy as sa",
     "import sqlalchemy as sa\nimport sqlmodel")

alembic("revision", "--autogenerate", "-m", "the heroes", "--rev-id", "0001")
alembic("upgrade", "head")
print("columns:", columns_of("hero"))


sqlmodel 0.0.42 | alembic 1.20.0 | the project: ['models.py']
$ alembic init migrations
    Creating directory migrations ...  done
    Creating directory migrations/versions ...  done
    Generating migrations/script.py.mako ...  done
    Generating migrations/env.py ...  done
    Generating migrations/README ...  done
    Generating alembic.ini ...  done
$ alembic revision --autogenerate -m 'the heroes' --rev-id 0001
    INFO  [alembic.autogenerate.compare.tables] Detected added table 'team'
    INFO  [alembic.autogenerate.compare.constraints] Detected added index 'ix_team_name' on '('name',)'
    INFO  [alembic.autogenerate.compare.tables] Detected added table 'hero'
    INFO  [alembic.autogenerate.compare.constraints] Detected added index 'ix_hero_age' on '('age',)'
    INFO  [alembic.autogenerate.compare.constraints] Detected added index 'ix_hero_name' on '('name',)'
    Generating migrations/versions/0001_the_heroes.py ...  done
$ alembic upgrade head
    INFO  [alembic.runtime.m

**1.** What Alembic has, and where the database is.


In [2]:
alembic("history")
alembic("current")


$ alembic history
    <base> -> 0001 (head), the heroes
$ alembic current
    0001 (head)


One revision, and the database is on it. `current` reads the `alembic_version` table; `history`
reads the files.


**2.** A motto for a team.


In [3]:
edit(SCRATCH / "models.py", "    headquarters: str = Field(max_length=60)",
     "    headquarters: str = Field(max_length=60)\n"
     "    motto: str | None = Field(default=None, max_length=80)")

alembic("revision", "--autogenerate", "-m", "a motto", "--rev-id", "0003")
revision("0003_a_motto.py")


$ alembic revision --autogenerate -m 'a motto' --rev-id 0003
    INFO  [alembic.autogenerate.compare.tables] Detected added column 'team.motto'
    Generating migrations/versions/0003_a_motto.py ...  done
def upgrade() -> None:
    """Upgrade schema."""
    # ### commands auto generated by Alembic - please adjust! ###
    op.add_column('team', sa.Column('motto', sqlmodel.sql.sqltypes.AutoString(length=80), nullable=True))
    # ### end Alembic commands ###


One `add_column`, with `AutoString` for the string and the `import sqlmodel` already in the file,
since the template has it.


**3.** The revision run.


In [4]:
alembic("upgrade", "head")
print("team columns:", columns_of("team"))


$ alembic upgrade head
    INFO  [alembic.runtime.migration] Running upgrade 0001 -> 0003, a motto
team columns: ['id', 'name', 'headquarters', 'motto']


The column is in the database, and `alembic_version` now names `0003`.


**4.** And undone.


In [5]:
alembic("downgrade", "-1")
print("team columns:", columns_of("team"))
alembic("current")

alembic("upgrade", "head")                                          # and put it back, for the next task


$ alembic downgrade -1
    INFO  [alembic.runtime.migration] Running downgrade 0003 -> 0001, a motto
team columns: ['id', 'name', 'headquarters']
$ alembic current
    0001
$ alembic upgrade head
    INFO  [alembic.runtime.migration] Running upgrade 0001 -> 0003, a motto


`downgrade -1` ran the revision's `downgrade`, which autogenerate wrote as the opposite of its
`upgrade`, and the database went back to `0001`. The revision file is still there, so
`alembic upgrade head` puts it back, which the last line does: autogenerate refuses to work while
the database is behind the revisions it already has.


**5.** A revision with nothing to do.


In [6]:
alembic("revision", "--autogenerate", "-m", "nothing to do", "--rev-id", "0004")
revision("0004_nothing_to_do.py")
(SCRATCH / "migrations" / "versions" / "0004_nothing_to_do.py").unlink()
print("deleted, and the history is:")
alembic("history")


$ alembic revision --autogenerate -m 'nothing to do' --rev-id 0004
    Generating migrations/versions/0004_nothing_to_do.py ...  done
def upgrade() -> None:
    """Upgrade schema."""
    # ### commands auto generated by Alembic - please adjust! ###
    pass
    # ### end Alembic commands ###
deleted, and the history is:
$ alembic history
    0001 -> 0003 (head), a motto
    <base> -> 0001, the heroes


The models and the database differ by the motto, which revision `0003` already describes, so
autogenerate found nothing new and wrote `pass`. Deleting a revision no database has run is safe.


**6.** Another database, from the migrations.


In [7]:
ini.write_text(ini.read_text().replace("sqlite:///heroes.db", "sqlite:///second.db"))
alembic("upgrade", "head")

run_python("""
from sqlmodel import Session, SQLModel, create_engine, select

from models import Hero, Team

engine = create_engine("sqlite:///second.db")
with Session(engine) as session:
    session.add(Team(name="Night Watch", headquarters="The Bell Tower", motto="We keep the hours"))
    session.commit()
    team = session.exec(select(Team)).one()
    print("wrote and read:", team.name, "|", team.motto)
""", name="use_second.py")


$ alembic upgrade head
    INFO  [alembic.runtime.migration] Running upgrade  -> 0001, the heroes
    INFO  [alembic.runtime.migration] Running upgrade 0001 -> 0003, a motto
wrote and read: Night Watch | We keep the hours


The second database was built by `upgrade head` and by nothing else, and the motto column is there
because revision `0003` is in the chain, whatever the first database happens to be on.

Last, this cell removes the scratch folder with the project and both databases in it:


In [8]:
shutil.rmtree(SCRATCH)

print("scratch still there:", SCRATCH.exists())


scratch still there: False


---

&#8592; **Back to:** [Migrations](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlmodel-deep-dive/12-migrations.ipynb)  &nbsp;&middot;&nbsp;  [SQLModel, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlmodel-deep-dive.html)
